# Notebook 11 — Severity Discrimination Analysis

This notebook measures how well COMET separates translations by error severity under native and romanised
scripts, and characterises the Marathi severity inversion — the only language where mean COMET
increases under romanisation yet reveals a perfectly monotonic severity-dependent reversal.

**Input:** per-language CSVs from `../../data/processed`, produced by earlier notebooks.

**Output:** three CSVs written to `../../results/tables/`.

## Setup

Import libraries, define file paths, and set column name constants that match the CSVs produced by the scoring
notebooks.

The severity order `SEVERITY_ORDER` follows the IndicMT Eval annotation scheme from Very Low (best
translation) through Very High (worst). Hindi lacks Very Low and Low annotations in the dataset;
its discrimination range is therefore computed from Medium → Very High.

The raw dataset stores per-error severity in five slot columns (`Error1_Severity` … `Error5_Severity`).
A single `Error_Severity` column is derived in the **Severity Derivation** cell below using worst-case rank.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/processed')
OUT_DIR  = Path('../../results/tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ['gujarati', 'tamil', 'malayalam', 'marathi', 'hindi']
ISO       = {'gujarati': 'GUJ', 'tamil': 'TAM', 'malayalam': 'MAL',
             'marathi': 'MAR', 'hindi': 'HIN'}

# COMET column names as written by 03_metric_scoring.ipynb
COL_COMET_NAT = 'comet'
COL_COMET_ROM = 'comet_rom'
# Derived severity column (built from Error1_Severity…Error5_Severity below)
COL_SEVERITY  = 'Error_Severity'
# Source slot columns present in the raw/processed CSV
SEVERITY_SLOTS = [f'Error{i}_Severity' for i in range(1, 6)]

SEVERITY_ORDER = ['Very Low', 'Low', 'Medium', 'High', 'Very High']
SEVERITY_RANK  = {s: i for i, s in enumerate(SEVERITY_ORDER)}

print('Config loaded.')

## Loading the Language Files

Read one CSV per language. Each file contains the original translation data plus COMET scores and
MQE error-severity annotations added by earlier notebooks.

All five DataFrames are collected into a `dfs` dictionary keyed by language name.

In [ ]:
dfs = {}
for lang in LANGUAGES:
    fp = DATA_DIR / f'{lang}_indicmt.csv'
    if fp.exists():
        dfs[lang] = pd.read_csv(fp)
        print(f'{ISO[lang]}: {len(dfs[lang])} rows  |  columns: {list(dfs[lang].columns[:10])}...')
    else:
        print(f'MISSING: {fp}')

## Severity Derivation

The IndicMT Eval dataset stores per-error severity in five slot columns
`Error1_Severity` through `Error5_Severity`. This cell collapses them into a single
`Error_Severity` column using **worst-case rank**: the most severe non-null label
across all five slots for each sentence.

Severity rank order (ascending → descending quality):
```
Very Low (0) < Low (1) < Medium (2) < High (3) < Very High (4)
```
If `Error_Severity` already exists in the CSV (written by a prior run), it is kept as-is.

In [ ]:
def derive_severity(df: pd.DataFrame) -> pd.Series:
    """Return worst-case severity across Error1_Severity…Error5_Severity."""
    present_slots = [c for c in SEVERITY_SLOTS if c in df.columns]
    if not present_slots:
        return pd.Series([np.nan] * len(df), index=df.index)
    # Map labels to numeric rank, take row-wise max, map back to label
    ranked = df[present_slots].apply(lambda col: col.map(SEVERITY_RANK))
    worst_rank = ranked.max(axis=1)  # NaN where all slots are NaN
    rank_to_label = {v: k for k, v in SEVERITY_RANK.items()}
    return worst_rank.map(rank_to_label)


for lang, df in dfs.items():
    iso = ISO[lang]
    if COL_SEVERITY in df.columns:
        print(f'{iso}: {COL_SEVERITY} already present — skipped')
    else:
        df[COL_SEVERITY] = derive_severity(df)
        n_derived = df[COL_SEVERITY].notna().sum()
        print(f'{iso}: derived {COL_SEVERITY} from slots ({n_derived} non-null)')
        dfs[lang] = df

    if COL_SEVERITY in df.columns:
        levels = sorted(df[COL_SEVERITY].dropna().unique(),
                        key=lambda x: SEVERITY_RANK.get(x, 99))
        print(f'  {iso} severity levels: {levels}')

## Severity Discrimination Collapse

Under native scripts, COMET correctly orders translations by error severity — mean score decreases
monotonically from Very Low to Very High errors. Romanisation collapses this discrimination range.

For each language the **discrimination range** is the difference between the Very Low and Very High
mean COMET scores (or Medium → Very High for Hindi, which lacks Very Low and Low annotations).
**Retention** is the romanised range expressed as a percentage of the native range.

Results are saved to `severity_discrimination_collapse.csv`.

In [ ]:
_RETENTION_REF = {'GUJ': 58.4, 'TAM': 37.7, 'MAL': 69.7, 'MAR': 28.9, 'HIN': 3.9}

sev_rows   = []
range_rows = []

print(f"{'Lang':>5}  {'Sev':>10}  {'N':>5}  {'COMET nat':>10}  {'COMET rom':>10}")
print('-' * 55)

for lang in LANGUAGES:
    iso = ISO[lang]
    df  = dfs[lang]

    if COL_SEVERITY not in df.columns or COL_COMET_NAT not in df.columns:
        print(f'{iso}: required columns missing')
        continue

    nat_means, rom_means = {}, {}
    for sev in SEVERITY_ORDER:
        sub = df[df[COL_SEVERITY] == sev]
        if len(sub) == 0:
            continue
        mn = sub[COL_COMET_NAT].mean() if COL_COMET_NAT in sub.columns else np.nan
        mr = sub[COL_COMET_ROM].mean() if COL_COMET_ROM in sub.columns else np.nan
        nat_means[sev] = mn
        rom_means[sev] = mr
        sev_rows.append(dict(
            lang=iso, severity=sev, N=len(sub),
            comet_mean_nat=round(mn, 2) if not np.isnan(mn) else None,
            comet_mean_rom=round(mr, 2) if not np.isnan(mr) else None
        ))
        print(f'{iso:>5}  {sev:>10}  {len(sub):>5}  {mn:>10.2f}  {mr:>10.2f}')

    top_sev = 'Medium' if iso == 'HIN' else 'Very Low'
    if top_sev in nat_means and 'Very High' in nat_means:
        nat_range = nat_means[top_sev] - nat_means['Very High']
        rom_range = rom_means.get(top_sev, np.nan) - rom_means.get('Very High', np.nan)
        retention = rom_range / nat_range * 100 if nat_range != 0 else np.nan
        flag = '\u2713' if abs(retention - _RETENTION_REF[iso]) < 2.0 else '~'
        range_rows.append(dict(
            lang=iso, ref_top=top_sev,
            native_range=round(nat_range, 2),
            romanised_range=round(rom_range, 2),
            retention_pct=round(retention, 1)
        ))
        print(f'  -> {iso} native range {nat_range:.2f} | rom range {rom_range:.2f} | retention {retention:.1f}%  {flag}')
    print()

pd.DataFrame(sev_rows).to_csv(OUT_DIR / 'severity_comet_by_level.csv', index=False)
pd.DataFrame(range_rows).to_csv(OUT_DIR / 'severity_discrimination_collapse.csv', index=False)
print(f'Saved: {OUT_DIR}/severity_comet_by_level.csv')
print(f'Saved: {OUT_DIR}/severity_discrimination_collapse.csv')

## Marathi Severity Inversion

Marathi is the only language where mean COMET increases under romanisation. Stratifying by severity
reveals a **perfectly monotonic reversal**: low-severity sentences (good translations) lose COMET
under romanisation; high-severity sentences (bad translations) gain COMET.

This analysis computes the per-severity mean delta (Romanised − Native) for Marathi, then calculates
the Spearman rank correlation between severity rank and mean delta to confirm the monotonic gradient.
A Wilcoxon signed-rank test on the full sentence-level distribution tests whether the overall shift
differs from zero.

Results are saved to `marathi_severity_inversion.csv`.

In [ ]:
_MAR_DELTA_REF = {
    'Very Low': -3.72, 'Low': -1.90, 'Medium': -0.73,
    'High': 1.08, 'Very High': 2.63
}

mar_rows = []

if 'marathi' in dfs:
    df = dfs['marathi']

    if COL_COMET_NAT in df.columns and COL_COMET_ROM in df.columns:
        both      = df[[COL_COMET_NAT, COL_COMET_ROM]].dropna()
        delta_all = both[COL_COMET_ROM] - both[COL_COMET_NAT]
        w_stat, w_p = stats.wilcoxon(delta_all)
        print(f'Marathi overall -- Wilcoxon W={w_stat:.0f}, p={w_p:.4f}')
        print(f'  mean delta = {delta_all.mean():.3f},  median delta = {delta_all.median():.3f}')
        print()

    print(f"{'Severity':>10}  {'N':>5}  {'Mean COMET delta (rom-nat)':>26}")
    print('-' * 47)

    sev_ranks, sev_deltas = [], []
    for rank, sev in enumerate(SEVERITY_ORDER):
        sub = df[df[COL_SEVERITY] == sev]
        if len(sub) == 0:
            continue
        if COL_COMET_NAT in sub.columns and COL_COMET_ROM in sub.columns:
            both_sub   = sub[[COL_COMET_NAT, COL_COMET_ROM]].dropna()
            mean_delta = (both_sub[COL_COMET_ROM] - both_sub[COL_COMET_NAT]).mean()
        else:
            mean_delta = np.nan
        flag = '\u2713' if (not np.isnan(mean_delta)
                          and abs(mean_delta - _MAR_DELTA_REF.get(sev, mean_delta)) < 0.5) else '~'
        mar_rows.append(dict(severity=sev, N=len(sub),
                             mean_delta_rom_minus_nat=round(mean_delta, 2)))
        sev_ranks.append(rank)
        sev_deltas.append(mean_delta)
        print(f'{sev:>10}  {len(sub):>5}  {mean_delta:>26.2f}  {flag}')

    valid = [(r, d) for r, d in zip(sev_ranks, sev_deltas) if not np.isnan(d)]
    if len(valid) >= 3:
        rr, dd = zip(*valid)
        spearman_r, spearman_p = stats.spearmanr(rr, dd)
        print(f'\nSpearman rho = {spearman_r:.3f},  p = {spearman_p:.4f}')

    pd.DataFrame(mar_rows).to_csv(OUT_DIR / 'marathi_severity_inversion.csv', index=False)
    print(f'\nSaved: {OUT_DIR}/marathi_severity_inversion.csv')
else:
    print('Marathi data not loaded.')

## Saving Results

Three files are written to `../../results/tables/`:

1. **`severity_comet_by_level.csv`** — mean COMET by severity level for both script conditions, all five languages.
2. **`severity_discrimination_collapse.csv`** — discrimination range and retention percentage per language.
3. **`marathi_severity_inversion.csv`** — per-severity mean COMET delta (Romanised − Native) for Marathi.

In [ ]:
print('=== Notebook 11 -- output manifest ===')
for f in sorted(OUT_DIR.glob('severity_*.csv')):
    print(f'  {f.name}')
for f in sorted(OUT_DIR.glob('marathi_*.csv')):
    print(f'  {f.name}')

## References

**Severity discrimination and Marathi inversion:**  
Anonymous (2026). *Under review.*

**Spearman rank correlation:**  
Spearman, C. (1904). The proof and measurement of association between two things. *The American Journal of Psychology*, 15(1), 72–101.

**Wilcoxon signed-rank test:**  
Wilcoxon, F. (1945). Individual comparisons by ranking methods. *Biometrics Bulletin*, 1(6), 80–83.

**COMET (neural MT metric):**  
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset:**  
Sai, A. B., Rao, S., Dabre, R., Kunchukuttan, A., & Khapra, M. M. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*, pp. 13831–13847. https://aclanthology.org/2023.acl-long.795